## Stage 1: PlasticESM, MLM post-training of ESM-C on the PETadex corpus

Lead     : `Purav Gupta`

Start    : `2026-10-16`

Status   : single-seed run complete and verified; dual-seed run in flight

Files    : `src/petabite/model_module/lora.py`, `src/petabite/mlm_finetune.py`, `scripts/`

This is the lab notebook for Stage 1. Stage 2, the activity labels and linear
probing, lives in the `igem-toronto` repo and is deliberately out of scope here.

## What Stage 1 is

Continue pretraining ESM-C 300M on the PETadex corpus with a masked-LM
objective, training LoRA adapters only and keeping the backbone frozen. The
result, PlasticESM, is the backbone Stage 2 puts an activity head on.

The objective is masked amino acid prediction on **sequence alone**. No
metadata of any kind enters the loss: not HMM family, not signal peptide, not
ProtParam, not taxonomy. That is deliberate, and the reasoning is in the
"metadata" section below.

## The repo did not work when this started

Four independent failures, each verified by running `origin/main` as it stood:

| | |
|---|---|
| `AutoTokenizer.from_pretrained` | `ValueError: Tokenizer class EsmcTokenizer does not exist` |
| `inject_fused_lora` | `RuntimeError: found no fused modules to wrap (qkv=0, ffn=0)` |
| `dump_pip_freeze` | `FileNotFoundError: 'uv'` |
| `config/mlm.yaml` | `eval_split: validation`, a split the corpus does not have |

The second is the one that mattered. `fused_lora.py` dispatched on
`_PyTorchLayerNormLinear` and `_PyTorchLayerNormMLP`; `esm` 3.4.1 renamed those
to `EsmcLayerNormLinear` and `EsmcLayerNormMLP`, so the tree walk matched
nothing. Earlier versions injected zero adapters and trained only the 30 PEFT
`out_proj` adapters while reporting a healthy loss curve. main's guard did at
least refuse to continue, but it blamed `transformer-engine`, which is not
installed.

Separately, saving was broken. PEFT and a hand-rolled mechanism both ran, and
`get_peft_model_state_dict` keeps only keys containing `lora_` **and**
`.default.`. The hand-rolled parameters matched the first test and failed the
second, so a checkpoint held 1,843,200 of 14,745,600 parameters at r=32.

`model_module/lora.py` replaces both with one mechanism over all four
projections per block, owning its own save and load, with three guards: raise
if nothing is adapted, prove each wrapper is bit-identical to the module it
replaced at init, and compare the saved file against the live model.

In [ ]:
# Reproduce the parameter arithmetic the brief reported, which is also the
# check that the adapter layout is understood correctly.
from transformers import AutoModelForMaskedLM
from petabite.model_module.lora import LoraConfig, apply_lora

model = AutoModelForMaskedLM.from_pretrained("biohub/ESMC-300M", trust_remote_code=True)
report = apply_lora(model, LoraConfig(r=32, alpha=64))

out_proj_only = sum(p.numel() for n, p in model.named_parameters()
                    if p.requires_grad and "out_proj" in n)

print(report.summary())
print(f"out_proj alone      : {out_proj_only:,}   (what PEFT used to save)")
print(f"fraction that was lost: {1 - out_proj_only / report.trainable_params:.1%}")

## The corpus

The published corpus is the 18,172,960 90%-identity centroids. One epoch over
all of them is about 57 h on an L40S, and most of the extra data is
near-duplicates, so Stage 1 trains on one member per **60%-identity** cluster
instead: 1,814,296 clusters, about 5.5 h per epoch.

Nothing in the parquet marks which 90%-centroid represents its 60% cluster, so
a rule had to be chosen, and the choice is not cosmetic. Picking each cluster's
largest member gives a subset that is **89.7% Logan** with a median length of
306, because Logan dominates the corpus and its ORFs are called stop to stop.
`build_c60_corpus.py` draws one member per cluster uniformly at random instead,
seeded, which is unbiased with respect to origin and length.

**81 clusters are excluded** because they contain a sequence from the PETai
label set. The exclusion uses the full 1,219-sequence label set rather than one
split's val/test, so it stays valid however Stage 2 re-splits later. No
downstream evaluation sequence is seen during pretraining in any form.

In [ ]:
import json
from pathlib import Path

for seed in (0, 1):
    path = Path(f"/project/6122149/purav/petabite-tokenized/c60-seed{seed}/manifest.json")
    if not path.exists():
        continue
    m = json.loads(path.read_text())
    print(f"seed {m['seed']}: {m['sequences']:,} sequences, {m['tokens']:,} tokens, "
          f"mean {m['mean_tokens']}, {m['clusters_excluded']} clusters excluded")

## Throughput

Measured on one L40S, 15 timed steps after 4 warmup, projected over the corpus.
Tokens per second rather than steps per second, because every knob being tuned
changes how many tokens a step contains.

| | step | tokens/s | peak mem | projected epoch |
|---|---|---|---|---|
| baseline: batch 8 x 1024, fp32, no TF32, unfused AdamW, SAE on, padded to longest | 0.783 s | 10,467 | 23.5 GB | 14.83 h |
| tuned: batch 32 x 512, bf16, TF32, fused AdamW, SAE off | 0.562 s | 26,610 | 31.2 GB | 5.83 h |
| plus length bucketing at batch 48 | 0.524 s | 27,497 | 44.1 GB | 5.65 h |

**2.63x**, 14.8 h to 5.65 h. Most of it is bf16 with TF32 and fused AdamW.
Truncating at 512 rather than 1024 is nearly free in coverage, since the corpus
median is 306 residues. Length bucketing is worth only 3% and costs 13 GB, so
the config keeps batch 32: at this point the run is compute-bound, not
padding-bound.

`compute_sae` is **not** a speed lever. ESM-C defaults it on and it runs a
sparse autoencoder head masked-LM training never reads, so it is switched off
on principle, but measurement found no difference beyond noise.

The single biggest remaining win would be `transformer-engine`, which fuses
LayerNorm+Linear and LayerNorm+MLP. At 27.5k tokens/s this is near 9% MFU on a
362 TFLOPS card. It is not taken because it swaps the fused modules for TE's
classes, which `lora.py` does not wrap, so `apply_lora` would raise. That is
the guard working as intended.

## A 24-hour stall that was not a hang

The first production run sat at `0/9423` for 13 minutes with the GPU at 0%, the
main process pegged on one core and all 8 dataloader workers idle.

`datasets` 5.0.0 changed `dataset[column]` to return a lazy `Column` rather
than a list. transformers reads the lengths that way for `group_by_length`, and
`LengthGroupedSampler` then indexes the result once per comparison while
sorting each megabatch. One lazy access costs 3.4 ms against 0.2 us for a list,
and the sampler needs roughly 23.5M of them: about **23 hours** of building the
batch order, before step one, inside a 12 h job.

Nothing errors. `Column` supports `len()` and indexing, so the sampler accepts
it and is merely 16,700x slower, which presents as a run that looks slow to
start rather than one that is broken.

`read_lengths` reads the Arrow column directly, 0.018 s for the same 1.8M
values. Because that only works in storage order, `split_tokenized` draws its
own permutation and applies it to both the rows and the lengths, and
`read_lengths` refuses a dataset carrying an indices mapping rather than
returning a silently misaligned array.

In [ ]:
import time
from petabite.data_module.tokenized import load_tokenized, read_lengths

corpus = load_tokenized("/project/6122149/purav/petabite-tokenized/c60-seed0")
lazy_column = corpus["length"]
print("type of corpus['length']:", type(lazy_column).__name__)

start = time.time()
_ = [lazy_column[i] for i in range(2000)]
lazy_per_access = (time.time() - start) / 2000

start = time.time()
lengths = read_lengths(corpus)
read_seconds = time.time() - start

start = time.time()
_ = [lengths[i] for i in range(2000)]
list_per_access = (time.time() - start) / 2000

print(f"lazy access  : {lazy_per_access * 1e3:.3f} ms each")
print(f"array access : {list_per_access * 1e6:.3f} us each  "
      f"({lazy_per_access / list_per_access:,.0f}x faster)")
print(f"read_lengths : {read_seconds:.3f} s for {len(lengths):,} values")
print(f"23.5M lazy accesses would be {23.5e6 * lazy_per_access / 3600:.1f} h")

## Run 1: single seed, one epoch

Job 6318656, 1,809,215 sequences, 9,423 steps, **5 h 17 m**.
https://wandb.ai/morphius/petabite-esmc/runs/4dwq33oj

| | |
|---|---|
| train loss | 1.8545 -> **1.678** |
| eval loss | 1.7168 -> **1.6836** |
| adapters | 240 tensors, 14,745,600 parameters, 59 MB |
| verification | completeness, trained, reload: all pass |

Most of the gain is in the first 10% of the epoch (1.854 to 1.715), then it
grinds slowly to 1.678. Eval tracks train with no divergence, so nothing is
overfitting, but it does say a second pass over the same data would buy little.

The up-projections moved from 3.3e-4 after 20 steps to 5.2e-3 at the end, so
gradients reached all 120 adapters, which is what `verify_checkpoint.py`
checks: an untrained adapter round-trips perfectly and is still worthless.

## Two runs collided, and what survived

Jobs 6318367 and 6318656 ran the identical configuration for 5 h 17 m each,
on kn089 and kn026, and **both wrote `outputs/plasticesm-c60-r32`**. Their final
eval losses differ only at the 4th decimal (1.68877 against 1.68867), which is
GPU nondeterminism.

The final artefact is clean: it is timestamped inside the surviving run's
finish, after the other had exited, and that run wrote and verified it as one
atomic sequence. An independent re-verification passed afterwards. The
intermediate checkpoints belonged to whichever job reached them last and were
deleted.

Two guards now exist in `train_mlm.sbatch`: the job id goes in `OUTDIR` by
default, and the script refuses to start when another `plasticesm` job is
queued or running.

## Run 2: two independent draws rather than two epochs

Because the corpus builder draws one member per cluster at random, `--seed 1`
picks a **different** member of most clusters. Loading both draws costs the same
wall time as two epochs and shows the model more distinct protein.

How much more is a real question rather than a rhetorical one, and the answer
is less than it first appears: **48.7% of the 60% clusters are singletons**, so
both seeds are forced to pick the same member there.

| | sequences seen | distinct |
|---|---|---|
| 2 epochs over one draw | 3.63M | 1,814,296 |
| seed0 + seed1 | 3.63M | **2,494,836** (1.38x) |

So a second draw adds 680,540 genuinely new sequences, not 1.8M. Still strictly
better than a second epoch, which adds none, but 1.38x rather than 2x.

In [ ]:
import collections
import pyarrow.parquet as pq

parquet = pq.ParquetFile(
    "/project/6122149/purav/petadex-corpus/centroids90/centroids90.parquet"
)
members_per_cluster = collections.Counter()
for group in range(parquet.num_row_groups):
    for cluster in parquet.read_row_group(group, columns=["c60_id"])["c60_id"].to_pylist():
        members_per_cluster[cluster] += 1

histogram = collections.Counter(members_per_cluster.values())
total = len(members_per_cluster)
# A second independent draw lands on a new member with probability 1 - 1/k.
expected_new = sum(n * (1 - 1.0 / k) for k, n in histogram.items())

print(f"60% clusters : {total:,}")
print(f"  singletons : {histogram[1]:,} ({100 * histogram[1] / total:.1f}%)")
print(f"expected new from a 2nd draw: {expected_new:,.0f}")
print(f"  one draw   : {total:,} distinct")
print(f"  two draws  : {total + expected_new:,.0f} distinct ({(total + expected_new) / total:.2f}x)")

## Stage 1's own question, answered

Whether two independent draws of the corpus beat one was the direct test of
whether more distinct sequence helps at this scale. Job `6334470` finished in
10h50m over 18,873 steps, exit 0, adapters written: 240 tensors and
14,745,600 LoRA parameters across all 30 blocks at r 32.

| run | job | steps | loss | perplexity |
| --- | --- | --- | --- | --- |
| single seed | `6318367` | 9,437 | 1.69644 | 5.4545 |
| single seed, again | `6318656` | 9,437 | 1.69640 | 5.4542 |
| two draws | `6334470` | 18,873 | **1.68426** | 5.3884 |

The accidental pair of identical single-seed runs turns out to be the most
useful number here. They agree to 4e-5 nats, so run-to-run noise at this
scale is negligible, and the dual-seed gain of 0.0122 nats is about 279 times
that spread. The gain is therefore real, and it is also 1.2% of perplexity
for 2.05 times the compute.

**The checkpoint was verified rather than assumed**, which was owed since the
run started. Job `6353892`. A healthy loss curve is not evidence that the file
on disk is usable: the original fused-LoRA bug saved 12.5% of the trained
tensors and the curve looked fine throughout. All three standard checks pass
on both runs, completeness, reloading into a fresh model, and every adapter
having moved off its zero initialisation, 0 dead of 240 in each.

Two things fell out of the comparison. The dual-seed adapters sit further from
initialisation, mean up-projection magnitude 7.47e-03 against 5.20e-03, which
is what twice the steps should do. And the two runs' weights genuinely differ,
smallest relative difference 0.304 and median 0.956, so the 0.0122 nats is
backed by different weights rather than by noise in the loss reporting.

Set against Stage 2, where neither backbone beats a single length column on
any target and no layer of either does either, that 0.0122 nats buys nothing
downstream. The honest reading is that MLM post-training on this corpus works
as intended, that the checkpoint it produced is sound, and that what limits
the project is neither of those things.

## Mistakes worth recording

Three, because each cost real time and each is now guarded against.

**A duplicate 5-hour run.** I relaunched without checking whether a production
run was already queued, so two identical jobs burned about 5.3 GPU-hours each.
`train_mlm.sbatch` now refuses to start alongside another `plasticesm` job.

**A corpus list that silently halved itself.** `sbatch --export` separates
variables with commas, so `--export=ALL,CORPUS="path0,path1"` arrives inside
the job as `CORPUS=path0`. Job 6334352 therefore trained one corpus while
reporting success; it was caught 3 minutes in only because the step count read
9,423 rather than the expected 18,873. The separator is now a colon, the
resolved list and its length are echoed, and a missing path exits before the
GPU is touched.

**A misdiagnosis of the stall.** The first hypothesis was that reading the
length column through a shuffle indices mapping was slow. Timing
`dataset["length"]` showed 0.00 s and appeared to clear it. That measured the
wrong half: the read is lazy, and the cost is in the 23.5M element accesses
afterwards.

## Why no metadata

ESM-C's corpus layers offer HMM family, signal peptide and ProtParam
descriptors at 100% coverage, and taxonomy and structure more sparsely. None of
the first three are used, for a reason worth stating precisely because it does
not generalise.

HMM family, SignalP and ProtParam are **cheap** functions of the sequence. A
300M model can plausibly learn them unaided, so supplying them adds an
inductive bias at best, not information.

Structure is an **expensive** function of the sequence: a 6B ESMFold2 trunk, 32
sampling steps, 3 loops. The 300M model cannot derive it at any amount of
training, so distilling it transfers computation the student cannot perform.
That asymmetry is why structure remains interesting and the rest does not.

Taxonomy was checked and rejected on coverage: **5.5%** on the 60% subset, not
the 54% an early biased sample suggested. The chain organism to BacDive to
growth temperature is therefore dead at the first link as well as the last.
`melting_point` and `thermostability` are null on all 307,155,746 corpus rows,
so there is no temperature target to distil from at all.

## Stage 2 evaluation: does any of this predict function?

Stage 1 produced an adapted backbone. This section is the measurement that says
whether it bought anything downstream, and it needs the whole section rather
than a number, because the first three answers it gave were wrong and finding
out why is most of the value.

The conclusion, stated first: **no feature set beats sequence length on any
target, substrate stratum or label regime, and PlasticESM never beats stock
ESM-C.** The useful result is not that null but the leakage measurement two
cells below, which says why published numbers on these targets look so much
better.

Two rules apply throughout. The unit of independence is the homology component,
never the sequence, because sequences inside a component share ancestry. And a
backbone result means nothing until it clears sequence length and amino acid
composition, because the very first probe found length beating both 960
dimensional backbones. The shared machinery is in
`src/petabite/eval_module/`.

| script | question |
| --- | --- |
| `scripts/embed_labels.py` | cache both backbones' embeddings once on a GPU |
| `scripts/embed_layers.py` | the same at six depths, via forward hooks |
| `scripts/eval_sweep.py` | every target and feature set, grouped 5-fold |
| `scripts/eval_power.py` | what effect size each label set could detect |
| `scripts/eval_leakage.py` | how much a random split would flatter us |
| `scripts/eval_conditions.py` | sequence property or assay protocol |
| `scripts/eval_robust.py` | stress whatever appears to win |
| `scripts/eval_metric.py` | pooled against fold-aware scoring |
| `scripts/eval_determinism.py` | does the score move with the BLAS threads |

Jobs, in order: `6335942` embeddings, `6336760` first sweep, `6340953`
conditions, `6341022` leakage, `6343514` and `6343616` depth, `6343981`
determinism, `6344624` metric, `6344807` corrected sweep, `6345366` corrected
leakage, `6345431` corrected protocol test.

## The measurement fault that invalidated three earlier answers

Worth putting before the results, because every number in the first version of
this notebook was affected and the same trap is easy to fall into again.

Scores were computed by concatenating the out-of-fold predictions into one
vector and correlating that against the target. This is the usual thing to do
and it is invalid under a strong ridge penalty. As the penalty grows, each
fold's predictions shrink toward *that fold's own training mean*, so the
concatenated vector ranks sequences partly by which fold they landed in. The
homology components here differ enough in mean that the effect dominates.

The controls were signalling it from the first run and were misread as a
finding. Sequence length is one column, so on the `p-np-c4` stratum its score
should be near its direct value. Computed within folds with no model involved
it is **+0.4157**. The pooled metric reported it between **+0.04 and -0.39**
depending on the penalty, and the first sweep printed **-0.062** for length on
`pet_release_conc` and **-0.193** on `km`.

So the headline "length beats the backbones" was right by accident, and the
later "the backbones beat length on p-np-c4" was the artefact talking. Three
retracted figures for that stratum: 0.404, then 0.30, then nothing.

The fix is to z-score each fold's predictions before pooling, which keeps the
ordering each model produced and discards only the fold-specific level and
scale. The per-fold mean is reported alongside as a check, and the two agree.
`stats.grouped_out_of_fold` does this by default now.

Three smaller faults were found on the way, each of which had bent a result:

- **Pre-norm against post-norm.** `TransformerStack.forward` returns
  `self.norm(x), x, ...`, so `last_hidden_state` is after a final LayerNorm
  while a hook on the last block sees it before. Column standard deviation
  26.8 against 0.008. The first layer sweep compared the two as if they were
  the same tensor.
- **Scale against penalty.** A ridge penalty is not scale invariant, and the
  cached features span 0.008 to 26.8. One alpha grid across them compares
  penalties rather than representations. Everything is standardised now.
- **A truncated grid and unstable selection.** The old grid stopped at 10,000,
  which is where the standardised optimum sat, and plain argmax over roughly
  17 training components swung the headline between 0.30 and 0.40. The grid
  reaches 1e6 and selection uses the one-standard-error rule.

Leave-one-component-out is dropped entirely. Its folds are often a single
sequence, which has no within-fold ordering, so it cannot be corrected and
every LOCO number from the first pass is unreliable.

## The sweep: nothing beats sequence length

Grouped 5-fold Spearman, fold-centred, standardised features, human-curated
labels, one value per sequence. Job `6344807`.

| target | n | comp | length | composition | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 157 | 18 | +0.055 | -0.056 | -0.387 | -0.331 | controls by 0.386 |
| `kcat` | 227 | 40 | **+0.343** | +0.133 | -0.025 | -0.217 | controls by 0.368 |
| `km` | 204 | 40 | +0.150 | +0.127 | -0.213 | -0.137 | controls by 0.287 |
| `kcat_over_km` | 216 | 42 | **+0.336** | +0.241 | +0.015 | +0.078 | controls by 0.258 |
| `melting_temperature` | 57 | 8 | +0.275 | -0.389 | +0.211 | -0.158 | controls by 0.064 |

With the 4,898 LLM-extracted labels folded in, which adds families as well as
measurements:

| target | n | comp | length | composition | stock | PlasticESM |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 188 | 26 | +0.060 | +0.091 | -0.112 | -0.165 |
| `relative_activity` | 82 | 22 | +0.175 | +0.147 | +0.081 | +0.097 |
| `kcat` | 263 | 49 | +0.272 | +0.031 | -0.041 | -0.082 |
| `km` | 226 | 46 | +0.003 | +0.138 | -0.013 | +0.066 |
| `kcat_over_km` | 229 | 45 | +0.260 | +0.313 | +0.052 | -0.009 |
| `weight_loss` | 91 | 27 | +0.090 | +0.128 | +0.074 | +0.044 |
| `melting_temperature` | 68 | 13 | +0.218 | -0.541 | -0.228 | -0.269 |

Twelve verdicts, twelve to the controls. The margins are larger than the first
pass reported, because length's real scores are around +0.34 rather than the
-0.19 to +0.28 the pooled metric printed.

The power simulation rules out the obvious excuse. At 80% power
`pet_release_conc` needs a true correlation of 0.5, so its null says nothing;
but `kcat`, `km` and `kcat_over_km` each detect 0.3, which is inside the range
published probes claim. For those three the null is real. The power machinery
correlates synthetic predictions directly against the target rather than
fitting a model, so it was not touched by the metric fault.

## Homology leakage sets the scale, and only for the high-capacity features

The most useful number here. The same targets, features and ridge, scored two
ways: a plain shuffled 5-fold, which is what sequence-to-function papers
commonly report, and the grouped 5-fold used everywhere else. Both sides are
standardised and fold-centred, so the comparison isolates the split. Job
`6345366`.

| target | n | comp | random stock | grouped stock | stock inflation | length inflation |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 157 | 18 | **0.626** | -0.387 | **+1.014** | -0.103 |
| `kcat` | 227 | 40 | **0.721** | -0.025 | **+0.746** | +0.177 |
| `kcat_over_km` | 216 | 42 | 0.570 | +0.015 | **+0.555** | +0.073 |
| `melting_temperature` | 57 | 8 | **0.741** | +0.211 | **+0.530** | -0.065 |
| `km` | 204 | 40 | 0.274 | -0.213 | **+0.487** | -0.123 |

Between 90% and 92% of the sequences carrying each target have a relative
elsewhere in the set, which bounds how much leakage can help, and the
embeddings nearly saturate that bound.

The split by feature set is the point. The 960-dimensional embeddings inflate
by 0.49 to 1.01; sequence length inflates by -0.12 to +0.18, which is nothing.
A high-capacity representation can memorise which family a sequence belongs to
and a single length column cannot, so the inflation is a property of
representational capacity rather than of the targets.

Read in the direction that matters: under the conventional protocol this
project would be reporting rho 0.72 on `kcat` and 0.74 on melting temperature,
numbers that look like a working model and would have gone on the wiki without
anyone being dishonest. With relatives removed from the training side the same
model on the same data is at zero.

Two consequences. The nulls above are not evidence that this approach
underperforms published work; they are what published work tends to look like
once homology is controlled. And the v2 split repair in the `igem-toronto`
repo, which cost a day and produced zero leakage at 90%, 50% and 30% identity,
is the only reason any of this can be stated.

## Is the target a property of the sequence, or of the assay?

The labels were the remaining suspect once noise and power were eliminated.
The hypothesis was that each target is mostly assay-protocol variance, since a
sequence carries a median of 4 measurements at different pH, temperature and
substrate and they are collapsed to one median. **It was wrong**, in all four
tests. `scripts/eval_conditions.py`, jobs `6340953` and `6344807`.

**1. Measurement noise is not the limit.** Variance decomposition, which
involves no cross-validation and so is untouched by the metric fault:

| target | within-sequence share | reliability of median | correlation ceiling |
| --- | --- | --- | --- |
| `pet_release_conc` | 18.2% | 0.947 | 0.973 |
| `kcat` | 18.5% | 0.815 | 0.903 |
| `kcat_over_km` | 27.5% | 0.725 | 0.852 |
| `km` | 35.0% | 0.650 | 0.806 |
| `relative_activity` | 50.4% | 0.496 | 0.704 |

Models scored at most +0.21 against ceilings of 0.81 to 0.97. Only
`relative_activity` is genuinely noise-limited, and it should be dropped as a
regression target for that reason.

**2. The protocol carries no detectable signal either.** A conditions-only
ridge over substrate, pH, temperature and log incubation time, scored out of
fold by component with fold centring (job `6345431`):

| target | measurements | rho | CI95 |
| --- | --- | --- | --- |
| `pet_release_conc` | 1448 | -0.234 | [-0.365, 0.215] |
| `kcat` | 544 | -0.157 | [-0.338, 0.084] |
| `kcat_over_km` | 481 | -0.117 | [-0.382, 0.238] |
| `km` | 454 | +0.105 | [-0.091, 0.393] |
| `relative_activity` | 1062 | +0.025 | [0.014, 0.840] |

Every interval spans zero, so the honest reading is no detectable protocol
signal. An earlier version of this notebook claimed the condition model was
*anti*-predictive at around -0.25 and built an argument about substrate and
family confounding on top of it. Those point estimates were produced by the
pooled metric and the claim is withdrawn.

**3. Holding the chemistry fixed does not rescue anything.** The pooled `kcat`
column does mix 40 substrates whose means span three orders of magnitude, from
-0.07 on MHET to +3.19 on acetylcholine, and `kcat` on acetylcholine is 50
measurements over 48 sequences in a single homology component. That is a real
defect in the label set. It is not what was stopping the models:

| variant | n | comp | best control | best model | verdict |
| --- | --- | --- | --- | --- | --- |
| `kcat` @ p-np-c4 | 73 | 22 | +0.279 | +0.282 | dead heat |
| `kcat` @ p-np-c2 | 48 | 20 | +0.407 | +0.067 | controls by 0.340 |
| `kcat_over_km` @ p-np-c4 | 73 | 22 | +0.362 | +0.332 | controls by 0.030 |
| `kcat_over_km` @ p-np-c2 | 48 | 20 | +0.202 | +0.153 | controls by 0.049 |
| `km` @ p-np-c4 | 73 | 22 | +0.439 | +0.226 | controls by 0.213 |
| `km` @ p-np-c2 | 48 | 20 | +0.030 | +0.241 | model by 0.211 |

The single model win, `km` on p-np-c2, is contradicted by the same target on
the other substrate, where the controls win by almost exactly as much. That
inconsistency is what noise looks like.

**4. Regressing the protocol out does not help.** Every residualised target is
null or negative for both backbones while the controls stay positive: `kcat`
-0.018 against +0.218, `kcat_over_km` -0.123 against +0.147, `km` -0.108
against +0.046, `pet_release_conc` -0.222 against +0.138. Residualising hurts
the embeddings specifically, which is unexplained and worth a look before
anyone builds on it.

In [ ]:
# The tables above, regenerated from the job outputs rather than retyped.
import json
from pathlib import Path

LOGS = Path("/project/6122149/purav/petabite/logs")
FEATURES = ("length", "composition", "stock", "plasticesm")
CONTROLS = ("length", "composition")
MODELS = ("stock", "plasticesm")

sweep = json.loads((LOGS / "corrected-6344807" / "sweep.json").read_text())
print("corrected sweep, grouped 5-fold, fold-centred")
header = f"{'target':22} {'n':>4} {'comp':>5}"
print(header + "".join(f"{name:>12}" for name in FEATURES) + "   verdict")
for target, result in sweep.items():
    if result.get("skipped"):
        continue
    scores = {name: result["features"][name]["kfold5"]["spearman"]
              for name in FEATURES}
    control = max(scores[name] for name in CONTROLS)
    model = max(scores[name] for name in MODELS)
    verdict = ("controls" if control >= model else "model")
    print(f"{target:22} {result['n_sequences']:>4} {result['n_components']:>5}"
          + "".join(f"{scores[name]:>+12.3f}" for name in FEATURES)
          + f"   {verdict} by {abs(control - model):.3f}")

print()
leakage = json.loads((LOGS / "corrected-6344807" / "leakage.json").read_text())
print("leakage: random split against grouped, and who it flatters")
print(f"{'target':22} {'random':>9} {'grouped':>9} {'inflation':>10}  feature")
for target, result in leakage.items():
    if result.get("skipped"):
        continue
    for name in ("stock", "length"):
        entry = result["features"][name]
        print(f"{target:22} {entry['random_spearman']:>+9.3f} "
              f"{entry['grouped_spearman']:>+9.3f} "
              f"{entry['inflation']:>+10.3f}  {name}")

## What the evaluation means for the project

**Stage 1 has no demonstrated downstream value.** PlasticESM does not beat
stock ESM-C anywhere, and on several targets it is worse. The adaptation
produced a backbone that works and not a backbone that helps. More epochs
would be spending compute on a question the evaluation cannot currently
answer in the affirmative.

**Neither backbone extracts usable activity signal from these labels.** Across
7 targets, 2 label regimes, 6 substrate strata and a residualised variant,
nothing clears sequence length. Three targets have adequate power to detect
0.3, so for them the null is a result rather than an absence.

**The leakage measurement is the publishable finding.** It is a methods result
rather than a model result: embeddings inflate by 0.49 to 1.01 under a random
split while sequence length does not inflate at all. It is also the strongest
justification for the v2 split repair.

**PET monomer release cannot adjudicate anything.** 157 sequences over 18
components, one holding 72% of the data, power threshold 0.5. It is the
project objective and the target least able to settle a question.

**The label set's limit is independent families, not measurements.** More
assays on the sequences already present will not help. Measurements on
unrelated enzymes will.

**Structure distillation was correctly deprioritised.** The bottleneck is not
representation quality, and a better encoder cannot fix a target that mixes 40
chemically unrelated reactions.

**Two label changes are indicated**, both in the `igem-toronto` release rather
than here: define `kcat`, `km` and `kcat_over_km` per substrate rather than
pooled, and drop `relative_activity` as a regression target given its 0.704
ceiling. Neither is expected to produce a positive result on its own; they
remove known defects so that a future negative means something.

## Search item 1: ranking variants inside one family. Negative.

The leakage result says the backbones recognise families. The obvious next
question is whether they can order the variants inside one, which is what a
protein engineering campaign actually needs and which family recognition
cannot help with. Job `6345802`, `scripts/eval_within_family.py`.

Component `00855c73` is the only place the data supports the question. It
holds 546 sequences and carries almost all of the dense coverage: 113 PET
release values spanning -3.30 to +4.52 in log10 micromolar, 76
`kcat_over_km`, 65 `kcat`, 52 `relative_activity`, 51 `km`, 47 melting
temperatures. Every earlier measurement collapsed this family to a single
unit of independence, discarding all of it.

Grouping inside the family is by identity cluster, not by sequence: 29
`cluster_90` groups among the 113 PET release sequences, 8 `cluster_50`
groups. Near-identical variants never straddle a split.

**The controls had to change.** Within a scaffold family a model can look
good by tracking drift from the wild type, and `length` is nearly
uninformative here: its direct within-fold correlation against PET release is
-0.146. So two variant-effect controls were added, identity to the family
medoid and substitution plus gap counts from it, both from BLOSUM62 global
alignment. The release records no wild type, so the medoid stands in for it.
Without these the bar would have been degenerate and almost anything would
have cleared it.

Seven hand checks on the alignment code passed before any score was computed:
self-identity exactly 1.0 with zero substitutions, a constructed point
mutation counted as exactly one substitution and zero gaps, a deletion
counted as a gap rather than a substitution.

### Within the family, holding out whole identity clusters

Fold-centred pooled rho, with the per-fold mean in brackets as the
conservative check.

| target | n | groups | length | composition | identity | mutations | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 113 | 29 | -0.003 | **+0.578** (0.196) | +0.449 | +0.462 | +0.406 (0.129) | +0.344 (0.056) | controls by 0.172 |
| `pet_release_conc` (cluster_50) | 113 | 8 | -0.015 | **+0.624** (0.450) | +0.172 | +0.393 | +0.519 (0.222) | +0.478 (0.187) | controls by 0.105 |
| `kcat_over_km` | 76 | 19 | +0.120 | +0.240 | +0.080 | +0.015 | -0.126 | -0.152 | controls by 0.366 |
| `kcat` | 65 | 20 | -0.098 | +0.042 | +0.083 | +0.019 | -0.045 | -0.069 | controls by 0.128 |
| `km` | 51 | 19 | +0.151 | +0.095 | -0.174 | +0.158 | -0.177 | -0.209 | controls by 0.335 |
| `relative_activity` | 52 | 14 | +0.278 | -0.212 | -0.073 | +0.284 | -0.034 | -0.104 | controls by 0.318 |
| `melting_temperature` | 47 | 10 | -0.067 | -0.137 | +0.140 | -0.085 | +0.160 | +0.260 (0.326) | model by 0.120 |

### Trained on every other component, predicting this family

| target | train n | train comps | length | composition | stock | PlasticESM |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 44 | 17 | +0.021 | -0.023 | **-0.434** (p 0.003) | **-0.392** (p 0.001) |
| `kcat_over_km` | 140 | 41 | +0.419 | +0.549 | +0.234 | +0.352 |
| `kcat` | 162 | 39 | +0.279 | +0.398 | +0.072 | -0.098 |
| `km` | 153 | 39 | +0.293 | +0.550 | -0.382 | -0.296 |

### Verdict against the acceptance test

Failed. The single apparent model win, PlasticESM on melting temperature at
+0.260 against a best control of +0.140, misses on three counts: the interval
[-0.098, 0.567] contains zero, the permutation p is 0.054 rather than below
0.01, and the power simulation puts 80% power at rho 0.5 for that target, so
a correlation of 0.26 could not have been detected reliably in the first
place. No replication on a second target either, since every other target
goes the other way.

Three things worth keeping from the negative.

**Composition is the best predictor of PET release inside the family**, at
+0.578 and +0.624 under the two groupings, beating both backbones. Amino acid
composition is a 20-number summary with no order information at all, so
whatever drives PET release in this variant series is visible without knowing
the sequence.

**Cross-family prediction is significantly backwards.** Both backbones
anti-predict PET release at -0.434 and -0.392 with permutation p of 0.003 and
0.001 while the controls sit at zero. This is the third time the embeddings
have gone significantly negative out of family, after the residualised
targets and several pooled-metric runs, and the pattern now survives the
metric fix. It is unexplained and it is the only consistent effect the
backbones produce.

**Pooled and per-fold scores disagree by a lot here**, for example composition
at +0.578 pooled against +0.196 per fold. Both are valid and the per-fold
mean is the conservative one. The verdict is the same either way, which is
why the table reports both rather than choosing.

## Search item 2: zero-shot likelihood. Negative.

Every negative so far came through a ridge fit, and three separate faults in
that machinery had already forced retractions: the pooled metric, the feature
scale, and the truncated penalty grid. So the next thing tried was a predictor
that fits nothing at all. The masked pseudo-log-likelihood of a sequence under
ESM-C is a single number per sequence with no training, no readout choice, no
penalty and no folds, which means none of those three faults can arise in it.
Jobs `6346834` and `6348599`, `scripts/zero_shot_likelihood.py`.

It is also the standard zero-shot variant-effect baseline in the protein
language model literature, and it had never been run here.

**The likelihood was verified before any score was read from it.** Each of
five real sequences beat a shuffled copy of itself, which preserves length and
composition exactly and destroys only the order, by margins of +1.1 to +1.8.
Mean real -1.3166 against mean shuffled -2.8623, with the uniform-vocabulary
floor at -3.4965. An independent longhand calculation sharing no code with the
function under test agreed to six decimals.

### Results

`pll` is the likelihood. Within-family numbers are fold-centred pooled rho
over identity clusters in component `00855c73`; cross-family numbers train on
every other component.

**Every interval quoted in this cell and the one above it is the component
bootstrap**, which a later cell shows excludes zero in 5% to 22% of null
draws against a nominal 5%. The permutation p is the criterion to read here.
Both verdicts are negatives, and a correctly wider interval only strengthens
a negative, so these were flagged rather than recomputed; the scripts now
report both intervals for any future run.

| target | setting | best control | pll stock | pll PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | within | +0.578 compos | -0.051 | -0.010 | controls |
| `kcat` | within | +0.083 identity | +0.171 | +0.158 | +0.088, p 0.147 |
| `kcat_over_km` | within | +0.240 compos | +0.001 | +0.021 | controls |
| `km` | within | +0.151 length | +0.078 | +0.108 | controls |
| `relative_activity` | within | +0.278 length | -0.159 | -0.135 | controls |
| `melting_temperature` | within | +0.140 identity | -0.092 | -0.089 | controls |
| `kcat` | cross | +0.398 compos | +0.385 | +0.350 | controls |
| `kcat_over_km` | cross | +0.549 compos | +0.215 | +0.190 | controls |
| `km` | cross | +0.550 compos | +0.362 | +0.360 | controls |
| `pet_release_conc` | within, cluster_50 | +0.624 compos | -0.032 | +0.018 | controls |
| `pet_release_conc` | cross | +0.021 length | +0.043 | +0.013 | +0.022, p 0.637 |

**Failed the acceptance test on every target.** The closest was `kcat` within
family at +0.171 against a best control of +0.083: a margin of +0.088 rather
than the required 0.10, a permutation p of 0.147 rather than below 0.01, and
an interval of [-0.275, 0.617] that contains zero.

Two things are worth keeping.

**The likelihood never anti-predicts.** Cross-family it reads +0.385, +0.215
and +0.362 on the three targets that support the comparison, while the fitted
embeddings on the same splits read +0.072, +0.234 and -0.382. The embeddings
swing across zero; the likelihood does not. That is the first evidence that
the anti-prediction is a property of the fitting rather than of the
representations, and it is followed up below.

**Composition keeps winning.** It is the best control in seven of the nine
rows, at +0.578 within family and +0.398, +0.549 and +0.550 across families.
A 20-number summary with no order information beats a 960-dimensional
representation of the same sequence, consistently.

### What the verification cost, and what it caught

Job `6346834` was cancelled at 57 minutes. It was 700 sequences into the first
of two backbones with 1h47m left and needed about 105 more, and
`np.savez_compressed` ran only after both backbones finished, so a timeout
would have discarded the completed work as well. The script now writes after
each backbone and sorts the queue longest first, since every masked copy is
padded to its own sequence's length.

The resubmitted job's gate then failed its own batching check, at -2.731472
against -2.734167 for a tolerance of 2e-3, and the job carried on and scored
the entire label set anyway. That is covered in the next section.

## Search item 3: the protocol as an input. Negative, and it makes things worse.

Everything up to here collapsed a sequence's measurements to one median and
discarded the protocol. That throws away rows, since a sequence assayed eight
times became one point, and it throws away what was done: which substrate, at
which pH and temperature, for how long. The pooled `kcat` column mixes forty
substrates, so a median over them is an average of things that are not
comparable. Job `6352307`, `scripts/eval_measurements.py`.

So this models the measurement. One row per measurement, the sequence's
features joined to its own protocol's features, the target being that
measurement's value.

**The headline is scored once per sequence, not once per measurement.** The
replicate counts are far too uneven to score rows: one `pet_release_conc`
sequence carries 227 of that target's 1448 measurements, and one component
carries 89% of them. A measurement-level correlation would largely be a
statement about how well one enzyme's replicates are ordered, which is the
same class of mistake as the fold-mean artefact. Collapsing first removes the
weighting and gives a number comparable with the aggregated sweep.

### The protocol as a feature hurts, everywhere

| target | meas | seq | comp | protocol only | length alone | length + protocol | best model alone | best model + protocol |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 1448 | 157 | 18 | -0.417 | +0.098 | +0.041 | -0.239 | -0.289 |
| `kcat` | 544 | 227 | 40 | -0.090 | **+0.416** | +0.042 | +0.287 | +0.268 |
| `kcat_over_km` | 481 | 216 | 42 | +0.002 | +0.382 | +0.116 | -0.093 | +0.169 |
| `km` | 454 | 204 | 40 | +0.027 | -0.042 | +0.011 | +0.011 | +0.010 |
| `melting_temperature` | 115 | 57 | 8 | -0.409 | -0.280 | -0.431 | +0.534 | +0.523 |

Adding the protocol to sequence length takes `kcat` from +0.416 to +0.042 and
`kcat_over_km` from +0.382 to +0.116. The protocol alone predicts nothing
anywhere, at -0.417 to +0.027. So the protocol columns are not a nuisance
variable the model can condition away; out of fold they are noise that
displaces a real feature, because substrate identity is confounded with
homology family and the confound reverses when families are held out.

This is the second time that has shown up. Residualising the protocol also
hurt, which is why `fit_conditions` fits in sample, and the reasoning there
was built on the pooled metric and withdrawn. The measurement-level version
reaches the same conclusion without that metric.

**The strongest homology-controlled result in the project is sequence length
on `kcat`, at +0.416**, with a permutation p of 0.002. It is a control, and
even it does not have an interval excluding zero once the interval is a
calibrated one: the jackknife gives [-0.166, 0.783] where the bootstrap gave
[0.018, 0.652]. That correction is the subject of a later cell.

## The depth sweep, redone on the corrected metric

The sweep had been run twice and both passes predate the metric fix, so
nothing had ever been validly claimed from it. The per-layer embeddings were
still cached, and `stock_post` in that cache is bit-identical to the `stock`
block used everywhere else, maximum difference 0.000e+00, so the two are
directly comparable. Job `6352640`.

Standardising is not optional here. Mean column standard deviation runs from
0.0085 at the post-norm output to 26.76 at block 29, a factor of 3150, so a
fixed penalty applied across layers compares amounts of shrinkage rather than
representations.

### The two best-powered targets, 40 and 42 components

Fold-centred pooled rho, standardised, one point per sequence.

| target | block | L5 | L10 | L15 | L20 | L25 | L29 | post | length | compos |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `kcat` | stock | +0.054 | +0.062 | -0.127 | +0.145 | +0.078 | +0.277 | +0.287 | **+0.416** | +0.225 |
| `kcat` | PlasticESM | +0.066 | -0.023 | -0.361 | +0.070 | +0.061 | -0.015 | +0.084 | | |
| `kcat_over_km` | stock | +0.136 | -0.367 | -0.045 | -0.089 | -0.205 | -0.213 | -0.205 | **+0.382** | +0.166 |
| `kcat_over_km` | PlasticESM | +0.174 | -0.375 | -0.207 | -0.117 | -0.224 | -0.026 | -0.093 | | |

**No layer of either backbone beats a single length column on either
target.** On `kcat` the two deepest readouts are the best of the fourteen at
+0.277 and +0.287, which at least matches the usual expectation that the last
layers carry the most about function. They still lose to one number per
sequence. On `kcat_over_km` every layer but the shallowest is negative, and
the shallowest two are the best, so even that ordering does not reproduce
across two targets.

So depth was not the problem, and the depth question is now closed rather than
pending.

## Why the embeddings anti-predicted. Resolved.

Three separate analyses had the backbones going significantly negative out of
family, most sharply at -0.434 and -0.392 on PET release with permutation p of
0.003 and 0.001. It survived the metric fix, it was the only consistent effect
the backbones produced, and it was recorded as unexplained.

It is an artefact of the ridge penalty, not a property of the
representations. Three independent pieces of evidence, from jobs `6352307`
and `6352414`.

**The score flips sign across the penalty grid.** For the stock embeddings on
`kcat`, the fold-centred score runs from -0.306 at alpha 1 to +0.292 at alpha
10,000. On PET release, -0.278 to +0.131. On `kcat_over_km`, -0.218 to +0.071.
A single length column, by contrast, holds at +0.415 to +0.416 across the
whole eight-order grid, because a one-column ridge's ordering cannot change
much when only its scale is shrunk.

**The score flips sign between raw and standardised features.** PlasticESM on
PET release reads -0.288 standardised and +0.090 raw.

**The direction is stable once the penalty is fixed.** Over 40 random
component-to-fold assignments it stays negative in 75% of draws, mean -0.137,
and dropping any one of the 18 components leaves it negative in 17 cases.

So the negative sign is real, reproducible, and a statement about where on the
shrinkage curve the penalty selection lands with 960 features and 18 to 42
independent units. It is not evidence that ESM-C encodes function backwards.
The zero-shot likelihood, which has no penalty at all, never goes negative,
which is the same conclusion reached from the other direction.

This also retires the earlier open question about residualisation hurting the
embeddings specifically: the embeddings' sign was never stable enough for that
asymmetry to have needed an explanation.

## The confidence interval was wrong, and it had manufactured a result

Item 3 produced the first thing in this project that looked like a finding.
The stock embeddings predicted melting temperature at +0.534 against a best
control of +0.085, with a permutation p of 0.002, a bootstrap interval of
[0.490, 0.942] that excluded zero, and a penalty profile that stayed positive
across all eight orders of magnitude of the alpha grid. The same features
sign-flip across that grid on every other target, so penalty stability was
new.

It also produced two things that could not be real. The protocol-only
baseline on that target scored **-0.409 with a permutation p of 0.003**, and
sequence length joined to the protocol scored **-0.431 with p 0.003**.
Controls are not supposed to be significantly anti-predictive. The redone
depth sweep on the same target then had the stock backbone swinging +0.47,
+0.43, +0.00, +0.25, -0.13, +0.46, +0.53 from block 5 to the post-norm
output, which is not a depth trend.

The fourth non-negotiable says that when a control moves like that, the metric
gets debugged rather than interpreted. So it was.

### What the pipeline reports when there is nothing to find

Job `6352882`. Hold the features and the group structure fixed, replace the
target with noise, and run the real scoring path, alpha selection and fold
centring included. 1600 draws on melting temperature across four feature
blocks and two kinds of null, 200 to 400 on the better-powered targets.

| | melting temp, 8 comp | `kcat`, 40 comp | `kcat_over_km`, 42 comp |
| --- | --- | --- | --- |
| rho under the null | mean within 0.014 of 0 | mean +0.009 | mean -0.006 |
| sd of rho | 0.137 to 0.163 | 0.081 | 0.069 |
| 95th percentile of \|rho\| | 0.258 to 0.316 | 0.160 | 0.126 |
| largest \|rho\| seen | 0.460 | 0.188 | 0.155 |
| permutation p < 0.01, nominal 1% | 0% to 4% | 0% to 2% | 0% to 3% |

Across all ten settings run, five feature-and-target pairs each under both
nulls, 1400 draws in total:

| criterion, nominal 5% | range over the ten settings | median |
| --- | --- | --- |
| **bootstrap excludes zero** | **5% to 22%** | **11.5%** |
| **jackknife excludes zero** | **1% to 9%** | **6%** |

Three conclusions, two of them corrections to what this notebook has been
reporting.

**The component bootstrap is anti-conservative by a factor of two to four.**
It resamples components with replacement, so a component drawn twice
contributes its rows twice, creating exact duplicate pairs that push a rank
correlation away from zero. Every interval reported in the cells above is
narrower in appearance than in meaning. `stats.jackknife_ci` deletes one
component at a time instead, which duplicates nothing, and works on the
Fisher z scale because its own gate caught a plain t interval returning
[0.309, 1.168], an upper bound no correlation can reach.

**The permutation test is roughly calibrated** and is the criterion that
binds from here.

**The controls that triggered this are explained.** Both -0.409 and -0.431
sit inside a null whose observed maximum is 0.460. They were noise reported
with an unreliable interval. The point estimates were never the problem.

Job `6353257` re-ran the whole measurement-level sweep with both intervals and
confirmed all 45 point estimates identical, so this is a correction to the
inference and not a new set of results. The consequence is blunt: **no result
in this project, control or model, has a calibrated interval excluding zero.**
Sequence length on `kcat`, the strongest of them at +0.416 with a permutation
p of 0.002, reads [-0.166, 0.783] by jackknife against [0.018, 0.652] by
bootstrap. The permutation test and the interval are not in conflict; they
answer different questions. There is a detectable association under component
permutation, and 40 components are not enough to pin its size down.

### The candidate does not survive the correct interval

Job `6353257`. Same predictions, same folds, both intervals side by side.

| | value |
| --- | --- |
| rho | +0.534 |
| bootstrap 95% | [0.490, 0.942] |
| **jackknife 95%** | **[-0.710, 0.969]** |
| permutation p | 0.002 |

The deleted-one estimates say why, and they are not what dropping a dominant
family was expected to show.

| component dropped | sequences held | rho without it |
| --- | --- | --- |
| `00855c73` | 47 | **+0.863** |
| `010eea6b` | 4 | +0.544 |
| six singletons | 1 each | +0.518 to +0.538 |

**`melting_temperature` has 57 sequences in 8 components: one holds 47, one
holds 4, and six hold a single sequence each.** Removing the dominant family
does not collapse the correlation, it raises it to +0.863. That is a jackknife
influence large enough to make the honest interval span zero.

So the acceptance test fails, and on the clause that matters rather than on a
technicality. A statistic that moves by 0.33 when one of eight components is
removed, where six of those components contribute one point each, is a
statement about these particular families rather than about melting
temperature. There is also no second thermostability target to replicate on,
and the result weakens from +0.534 to +0.147 without standardising.

**Reported as a negative.** The data that would change it is the only thing
worth naming: melting temperatures measured across more homology families.
Eight components of which six are singletons cannot carry this claim however
it is analysed, and the null calibration puts the 95th percentile of pure
noise at 0.316 on this target against 0.126 on `kcat_over_km`.

## Search item 4: classify instead of regress. Negative, and weaker.

A `kcat` column pooled over forty substrates is not one quantity, so its
numeric value carries the protocol as much as the enzyme, and item 3 showed
that handing the model the protocol as an input makes things worse. Asking
instead whether an enzyme is in the better half of those assayed on the *same*
substrate moves that heterogeneity into the label definition, where it costs
nothing. Job `6353282`, `scripts/eval_classification.py`.

The threshold is taken within substrate, never globally, because substrates
differ in their typical values and a global median split would partly label
the substrate rather than the enzyme. The metric is the area under the ROC
curve, rank based like the Spearman correlation used elsewhere, on
fold-centred predictions, with the jackknife interval. The null is 0.5.

| target | seq | active | comp | substrates | length | compos | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 157 | 80 | 18 | 3 | 0.443 | 0.488 | 0.332 | 0.417 | controls 0.071 |
| `kcat` | 226 | 119 | 40 | 40 | **0.600** | 0.567 | 0.569 | 0.509 | controls 0.031 |
| `km` | 203 | 117 | 40 | 31 | 0.556 | 0.529 | 0.490 | 0.480 | controls 0.066 |
| `kcat_over_km` | 215 | 116 | 42 | 34 | 0.566 | 0.439 | 0.558 | 0.486 | controls 0.008 |

**Failed, and the framing is weaker than the regression it replaced.** The
best number anywhere is length on `kcat` at an area of 0.600, permutation p
0.006, and even that has a jackknife interval of [0.455, 0.729] which
contains 0.5. The same feature on the same target gives a Spearman of +0.416
as a regression, which is a much stronger statement.

The reason is visible in the substrate counts. `kcat` has 226 sequences
spread over **40 substrates**, about five or six each, and many hold only two.
A median split inside a two-sequence substrate labels one active and one
inactive by construction, whatever the values are, so the thresholding adds
noise rather than removing protocol variance. The framing is sound; the data
does not have enough sequences per substrate to use it.

## Search item 5: fit every target at once. Negative, and the premise was wrong.

This was the last idea in the search space that attacked the binding
constraint rather than working around it. Every single-target analysis is
limited by its component count, and the null calibration priced that
precisely: the 95th percentile of pure noise rises from 0.126 on
`kcat_over_km` with 42 components to 0.316 on `melting_temperature` with 8.
The hope was that the union of all targets would reach many more components
than any one of them, so a shared direction would be estimated from far more
independent units. Job `6353513`, `scripts/eval_multitask.py`.

One row per sequence and target, each target z-scored on its own because
Celsius and log10 `kcat` cannot share a scale, features being the sequence
block joined to a target one-hot so each target keeps its own intercept.
Centring is per fold **and target**, since a fold holds rows from several
targets at different levels and centring the fold as a whole would leave that
contrast in. The gate checks exactly this: a pure per-target offset must
centre away to nothing while the ordering inside a target survives.

**The premise did not hold.** The stacked table reaches 56 components, not
the many more the idea needed.

| target | sequences | components |
| --- | --- | --- |
| `kcat_over_km` | 216 | 42 |
| `kcat` | 227 | 40 |
| `km` | 204 | 40 |
| `pet_release_conc` | 157 | 18 |
| `melting_temperature` | 57 | 8 |
| `relative_activity` | 55 | 4 |
| **union** | **428** | **56** |

The targets concentrate on the same families, so pooling six of them buys 14
components over the best single one. The label set has 184 components in
total, but the ones carrying a named, normalised target are 56 of them.

### Results

| target | length | compos | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- |
| `kcat` | +0.349 | +0.128 | +0.262 | +0.117 | controls 0.087 |
| `kcat_over_km` | **+0.345** | +0.189 | +0.193 | -0.029 | controls 0.152 |
| `km` | -0.005 | -0.052 | -0.098 | +0.055 | model 0.060 |
| `melting_temperature` | -0.290 | +0.399 | +0.266 | -0.565 | controls 0.133 |
| `pet_release_conc` | -0.093 | -0.105 | +0.294 | +0.273 | model 0.387 |
| `relative_activity` | +0.234 | +0.108 | +0.001 | +0.156 | controls 0.078 |

**Failed.** The two apparent model wins do not survive their own statistics.
On `pet_release_conc` stock reads +0.294 with a permutation p of 0.051 and a
jackknife interval of [-0.629, 0.873]; on `km` the margin is +0.060, below the
required 0.10, at p 0.49.

Sharing also costs the controls. Length falls from +0.416 to +0.349 on `kcat`
and from +0.382 to +0.345 on `kcat_over_km`, because one shared direction has
to compromise across six targets that do not agree.

A useful sanity check fell out of it. The target one-hot on its own scores
nothing at all, correctly: its predictions are constant within a target, so
per-target centring reduces them to zero and the correlation is undefined
rather than small. A baseline that should carry no information carries none.

**And the single most defensible statistical claim in this project turned up
here.** Length against `kcat_over_km`, at +0.345 with a permutation p of 0.001
and a jackknife interval of **[0.12, 0.535] that excludes zero**. It is a
control.

It also replicates, which nothing else in this project does. Item 7 scores the
same pair in the ordinary aggregated setting, one median per sequence and no
target sharing, and gets +0.350 with p 0.001 and a jackknife of [0.152,
0.522]. Two framings, two fits, the same answer. So the one claim here that
meets the acceptance test's statistical bar and replicates is that longer
PETases have higher kcat/Km.

## Search item 6: the structures are not there

The search space listed structure features from the ESMFold2 structures
"already computed". They are not computed, and the annotations that do exist
locally cannot reach the labelled sequences.

What is actually in the corpus:

| layer | rows | granularity | usable here |
| --- | --- | --- | --- |
| `afdb_structures` | 308,394 | per ORF | manifest of S3 keys only, no coordinates |
| `component_cath` | 42 | **per component** | cannot work, see below |
| `catalytic_domains` | 8,246,431 | per ORF | 4.7% coverage |
| `biochem_mature` | 42,522,458 | per ORF | 4.7% coverage |

`component_cath` is worth a sentence because it looks promising and is
mathematically disqualified. It has one row per homology component, so its
value is constant inside a component. The homology component is the unit the
folds are built from, so a component-constant feature has zero variance
within any fold, and after fold centring it contributes exactly nothing. It
can only express the contrast between families, which is the leakage already
measured, and it can never rank variants inside one.

The other two are per sequence and would be the right kind of feature,
catalytic site identities especially, since a disrupted catalytic triad is a
mechanistic rather than a statistical reason for an enzyme to be dead. They
are unreachable for a different reason, in the next cell.

## Search item 7: a learned head. Negative, as predicted, for the stated reason.

Half of this item was not testable. Learned pooling needs per-token hidden
states and only the mean-pooled vectors were cached, so recomputing them is
about 1.1 billion floats, a job rather than an analysis.

The other half was, and the prediction was written into the script before the
run: a head does worse than the ridge, because the ridge alpha profile for
these embeddings peaks at the most regularised end of an eight-order grid.
Stock on `kcat` reads -0.306 at alpha 1 and +0.292 at alpha 10,000. A score
that improves as capacity is removed says the data supports almost none, and
a head adds capacity in that direction. Job `6353593`, three widths by three
penalties by three seeds.

| target | features | ridge | ridge p | ridge jackknife | heads, mean over seeds | worst seed sd |
| --- | --- | --- | --- | --- | --- | --- |
| `kcat` | length | **+0.342** | 0.002 | [-0.128, 0.687] | +0.214 to +0.336 | 0.181 |
| `kcat` | compos | +0.133 | 0.115 | [-0.023, 0.282] | -0.003 to +0.247 | 0.153 |
| `kcat` | stock | -0.025 | 0.721 | [-0.206, 0.158] | -0.012 to +0.088 | 0.156 |
| `kcat` | PlasticESM | -0.217 | 0.009 | [-0.396, -0.022] | +0.006 to +0.080 | 0.146 |
| `kcat_over_km` | length | **+0.350** | 0.001 | **[0.152, 0.522]** | +0.133 to +0.350 | 0.117 |
| `kcat_over_km` | compos | +0.240 | 0.008 | [-0.090, 0.522] | -0.172 to +0.018 | 0.115 |
| `kcat_over_km` | stock | +0.015 | 0.858 | [-0.271, 0.299] | -0.167 to -0.019 | 0.090 |
| `kcat_over_km` | PlasticESM | +0.078 | 0.387 | [-0.306, 0.440] | -0.167 to -0.069 | 0.112 |

**Failed, and no head beats the ridge on the features that matter.** On
length the best head matches the ridge and never exceeds it. On the embeddings
every head sits within 0.09 of zero on `kcat` and is negative throughout on
`kcat_over_km`.

The seed spread is the other half of the answer. A head's score moves by up to
0.181 between three initialisations of the same configuration, which is larger
than most of the differences anyone would want to interpret. The caveat
recorded before the run cuts the same way: the head's early stopping uses a
random split of the training rows, so its stopping point is chosen with
within-component leakage the ridge's nested alpha selection does not have. The
head was given an advantage and still lost.

One incidental confirmation. Heads move PlasticESM on `kcat` from the ridge's
-0.217 to about +0.05. A different implicit amount of regularisation lands in
a different part of the shrinkage curve, and the sign follows it. That is the
same conclusion the alpha profile reached, arrived at by a method that never
touched the alpha grid.

## Why none of it worked: the corpus and the labels are different distributions

This is the finding the five negatives were pointing at, and it was found by
trying to join the labels to the corpus annotations rather than by any of the
modelling.

**Only 53 of the 1137 labelled sequences, 4.7%, appear in the PETadex
pretraining corpus at all.** For the 721 human-curated labels, the best ones,
it is 12 sequences, or 1.7%.

Checked two independent ways, because it is a large claim. The release's
`seq_md5` is confirmed to be the MD5 of the sequence itself, and matching on
exact sequence content against the corpus's 1,158,140 distinct sequences
gives the same 53.

| provenance | labelled sequences | in the corpus |
| --- | --- | --- |
| human | 721 | 12 (1.7%) |
| llm | 358 | 35 (9.8%) |
| mixed | 58 | 6 (10.3%) |
| **total** | **1137** | **53 (4.7%)** |

And the label set is concentrated to a degree that makes the rest of it
inevitable:

| component | labelled sequences | share | in the corpus |
| --- | --- | --- | --- |
| `00855c73` | 546 | **48.0%** | 10 of 546 |
| `00c28e4a` | 63 | 5.5% | |
| `03589827` | 54 | 4.7% | |
| `05ce4f9d` | 28 | 2.5% | 15 of 28 |
| remaining 180 | 446 | 39.2% | |

### What this explains

Stage 1 post-trained ESM-C with LoRA on 1.16 million natural metagenomic
ORFs. Stage 2 evaluated it on a set that is 48% one scaffold family and 95%
absent from that corpus, which is to say mostly engineered variants from the
literature. The adaptation and the evaluation are on different distributions,
and nothing in an MLM objective over natural diversity is about telling a
point mutant from its wild type.

Every specific pattern in the cells above follows from that.

**Composition beats the backbones inside the family**, at +0.578 against
+0.406. Within a series of point mutants of one scaffold, the composition
vector is close to a direct encoding of which mutations are present, so a
20-number summary is not a weak baseline here, it is nearly the answer key.

**Length is the strongest predictor across families**, at +0.416 on `kcat`.
Across families length tracks family identity, and families differ in `kcat`.

**Within-family ranking fails for everything.** That needs variant-effect
sensitivity, which is the one thing this adaptation could not have acquired.

**The zero-shot likelihood fails inside the family specifically**, at -0.051
on PET release, while working across families at +0.385 on `kcat`. A
likelihood scores how natural a sequence looks. Engineered high performers are
selected to beat natural enzymes, so looking natural and working well pull
apart exactly where the data is densest.

**Stage 1's 0.0122 nat improvement buys nothing downstream**, because it is an
improvement at modelling a distribution the evaluation set is not drawn from.

### What data would change the answer

Named specifically, since that is the deliverable for a negative result.

1. **Measurements on more homology families.** 48% of labels in one component,
   and `melting_temperature` having 6 of its 8 components as single
   sequences, is why the honest intervals span zero. The null calibration puts
   the 95th percentile of pure noise at 0.316 on that target against 0.126 on
   `kcat_over_km`, purely from the component count.
2. **A deep mutational scan of one scaffold under one assay.** This is the
   regime where variant-effect models work, and it is the question a protein
   engineering campaign actually asks. The current data has the variants but
   spread over incomparable assays.
3. **Post-training on the engineered-variant distribution** rather than on
   natural diversity, if the goal is to rank variants. The corpus is the wrong
   corpus for this evaluation, which is a statement about the pairing and not
   about either piece on its own.

Nothing here says ESM-C carries no information about PET hydrolysis. It says
this label set cannot demonstrate it, that the controls it has to beat are
unusually strong for structural reasons, and that the pretraining corpus and
the labels were never a matched pair.

## Where this leaves the project

Every item in the search space has been tested or answered. All seven are
negative, and the reason they are all negative is the same one, established in
the cell above rather than guessed at.

| # | idea | outcome | job |
| --- | --- | --- | --- |
| 1 | rank variants inside one family | negative | `6345802` |
| 2 | zero-shot ESM-C pseudo-likelihood | negative | `6348599` |
| 3 | measurement-level, protocol as input | negative, and it hurts | `6352307` |
| 4 | classify active against inactive | negative, weaker than regression | `6353282` |
| 5 | multi-task across targets | negative, premise wrong | `6353513` |
| 6 | structure features | unavailable, 4.7% coverage | n/a |
| 7 | learned head instead of ridge | negative, as predicted | `6353593` |

**No representation of these sequences beats sequence length or amino acid
composition on any target, under any of the seven framings, at any depth of
ESM-C, with or without the assay protocol as an input.**

Exactly one claim in this project clears the acceptance test's statistical bar
and replicates, and it is a control: **longer PETases have higher kcat/Km.**

| setting | rho | p | jackknife 95% |
| --- | --- | --- | --- |
| aggregated, single target | +0.350 | 0.001 | [0.152, 0.522] |
| multi-task over six targets | +0.345 | 0.001 | [0.120, 0.535] |

Two framings, two fits, the same answer, with a calibrated interval excluding
zero in both. Nothing else in the project replicates at all.

### What was wrong with the evaluation, and is now fixed

Worth listing separately, because four of these invalidated results that had
already been written down.

1. **Pooled out-of-fold predictions** without fold centring ranked sequences
   by fold membership. Fixed, and it retracted a +0.404.
2. **The ridge penalty is not scale invariant** across a 3150-fold range of
   feature scale. Fixed by standardising with the scaler fitted on training
   rows only.
3. **The alpha grid was truncated** at exactly the value where the optimum
   sat, and argmax selection swung a headline between 0.30 and 0.40. Fixed by
   extending the grid to 1e6 and selecting by the one-standard-error rule.
4. **The component bootstrap was anti-conservative** by a median factor of
   2.3, excluding zero in 5% to 22% of null draws against a nominal 5%. Fixed
   by the jackknife, which measures 1% to 9%. This one changed no point
   estimate and every inference.
5. **A verification gate that failed did not stop its job.** Every sbatch
   file set `-u` and `-o pipefail` but never `-e`, so the gates were
   decorative. Fixed, and it has caught two real faults since.

### Still open

- **`relative_activity` and `weight_loss` are unscoreable**, at 4 and 2
  homology components. They are in the release and cannot carry any claim.
- **The 4.7% corpus overlap should be reconciled with issue 117**, which
  measured pretraining overlap against the v2 evaluation folds in the `petai`
  repo. These two numbers are about the same thing and should agree.
- **Learned pooling was not tested**, only a small head. It needs per-token
  hidden states, about 1.1 billion floats to recompute. It is also the variant
  the alpha profile most strongly predicts against, since it adds parameters
  where the evidence says the data supports almost none.
- **Whether a corpus of engineered variants would change the answer.** This is
  the one experiment the diagnosis actually points at, and it is a data
  acquisition rather than an analysis.

### What not to do next

Not another framing of this label set against these controls. Seven have been
tried and the failure mode is structural: 48% of the labels are one scaffold
family, 95% of the sequences are absent from the pretraining corpus, and the
targets that are scoreable reach 56 homology components between them. The
arithmetic of the null calibration says what that costs, and no choice of
model, readout, metric or framing recovers it.